# Module 3.5 · Solutions
Each SQL answer is paired with its pandas verification — two independent routes to the same number.

In [ ]:
import os, pandas as pd
from sqlalchemy import create_engine, text
engine = create_engine(os.environ.get("COURSE_DB_URL") or "sqlite://")
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
cl = pd.read_csv(BASE + "client_book.csv"); tx = pd.read_csv(BASE + "messy_transactions.csv")
if str(engine.url) == "sqlite://":
    cl.to_sql("clients", engine, index=False); tx.to_sql("transactions", engine, index=False)
def q(sql):
    with engine.connect() as conn: return pd.read_sql(text(sql), conn)

In [ ]:
# Ex1
print(q("""SELECT client_id, aum_inr FROM clients
WHERE risk_profile='Aggressive' AND city='Mumbai' AND aum_inr > 5000000
ORDER BY aum_inr DESC""").head())
# pandas twin
p = cl[(cl.risk_profile=='Aggressive') & (cl.city=='Mumbai') & (cl.aum_inr>5e6)]
print("pandas count matches:", len(p))

In [ ]:
# Ex2
print(q("""SELECT relationship_manager, COUNT(*) n, ROUND(AVG(churned),3) churn
FROM clients GROUP BY relationship_manager HAVING COUNT(*) >= 30 ORDER BY churn DESC""").head())
g = cl.groupby('relationship_manager').agg(n=('client_id','count'), churn=('churned','mean'))
print(g[g.n>=30].sort_values('churn', ascending=False).round(3).head())

In [ ]:
# Ex3
print(q("""SELECT c.city, COUNT(DISTINCT t.customer_id) transactors, ROUND(SUM(t.amount_inr),0) spend
FROM clients c LEFT JOIN transactions t ON c.client_id = t.customer_id AND t.amount_inr > 0
GROUP BY c.city ORDER BY spend DESC"""))

In [ ]:
# Ex4
print(q("""SELECT city, client_id, aum_inr FROM (
  SELECT city, client_id, aum_inr,
         RANK() OVER (PARTITION BY city ORDER BY aum_inr DESC) rnk
  FROM clients WHERE aum_inr IS NOT NULL) x WHERE rnk = 1"""))

In [ ]:
# Ex5 - the five business questions (SQL + pandas verification pattern shown for Q2 and Q4)
# Q2: share of AUM per city
print(q("""SELECT city, ROUND(100.0 * SUM(aum_inr) / SUM(SUM(aum_inr)) OVER (), 1) AS pct
FROM clients WHERE aum_inr IS NOT NULL GROUP BY city ORDER BY pct DESC"""))
pdv = (cl.groupby('city').aum_inr.sum() / cl.aum_inr.sum() * 100).round(1).sort_values(ascending=False)
print("pandas twin:\n", pdv)

# Q4: SIP-active clients with zero transactions
print(q("""SELECT COUNT(*) FROM clients c LEFT JOIN transactions t ON c.client_id=t.customer_id
WHERE c.sip_active = 1 AND t.txn_id IS NULL"""))
has_txn = set(tx.customer_id)
print("pandas twin:", ((cl.sip_active==1) & (~cl.client_id.isin(has_txn))).sum())